<a href="https://colab.research.google.com/github/jessieliang-0720/git-tutorial/blob/main/prepare_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 準備示範資料集（跑一次就好）

這個 notebook 會**自動幫你建好整包照片**，讓你不用先準備照片就能 demo 刷臉門禁。

執行完之後，你的資料夾會長這樣：

```
AI_door/
├── faces/
│   ├── Colin_Powell/    ← 「要放行的人」約 25 張（同一個人）
│   └── others/          ← 約 40 張，來自 20 個不同的人
└── test/
    ├── demo.jpg         ← Colin Powell，但訓練時沒看過這張
    └── stranger.jpg     ← Tony Blair，完全沒出現在訓練資料裡
```

---

## ⚠️ 先講清楚三件事

**1. 這裡面沒有「你」的照片。**
我不可能有你的照片。所以這份示範資料是拿一個公眾人物來當「要放行的人」

**2. 這是真人的照片。**
用的是 **LFW（Labeled Faces in the Wild）**，馬薩諸塞大學做的人臉辨識標準資料集，
內容是新聞照片裡的公眾人物。**這是學術界公開、免費、專門給人臉辨識研究用的資料集**，

**3. 為什麼「我」和「其他人」都用同一個資料集？**
這很重要。如果「要放行的人」用真實照片、「其他人」用 AI 生成的假臉，
模型會學到「怎麼分辨真照片和生成圖」

---

### 下載資訊

LFW 大約 **200MB**，第一次跑要等 **1～3 分鐘**。
之後 sklearn 會把它快取起來，重跑就很快。


---
## 1️⃣ 設定資料夾

**跟主 notebook 用同一個設定** —— 資料夾名稱要一樣，不然主 notebook 會找不到。


In [2]:
import os, glob, shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt

#@title 📁 設定資料夾位置 { display-mode: "both" }

#@markdown **照片要存在哪裡？**
#@markdown * `drive` — Google 雲端硬碟：**永久保存（推薦）**
#@markdown * `colab` — Colab 暫存：關掉分頁就消失
#@markdown * `local` — 本機資料夾：VS Code / Jupyter
storage = "drive"  #@param ["drive", "colab", "local"]

#@markdown **資料夾名稱**（會建在上面選的位置底下）
folder_name = "AI_door"  #@param {type:"string"}

if storage == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.path.join('/content/drive/MyDrive', folder_name)
elif storage == 'colab':
    ROOT = os.path.join('/content', folder_name)
else:
    ROOT = os.path.abspath(folder_name)

FACES_DIR = os.path.join(ROOT, 'faces')
TEST_DIR = os.path.join(ROOT, 'test')
os.makedirs(FACES_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print('📂 資料會建在：')
print(f'   {FACES_DIR}')
print(f'   {TEST_DIR}')

Mounted at /content/drive
📂 資料會建在：
   /content/drive/MyDrive/AI_door/faces
   /content/drive/MyDrive/AI_door/test


---
## 2️⃣ 下載 LFW

第一次執行要等 1～3 分鐘（約 200MB）。跑完會列出可以選的人。

In [ ]:
from sklearn.datasets import fetch_lfw_people

print('正在下載 LFW…（第一次約 1~3 分鐘，之後會用快取）\n')

# slice_ 比預設寬一點 → 臉的周圍留多一些背景，人臉偵測器比較好抓
# color=True → 要彩色的（MobileNetV2 吃 3 通道）
lfw = fetch_lfw_people(
    min_faces_per_person=30,
    color=True,
    funneled=True,
    slice_=(slice(48, 202), slice(48, 202)),
    resize=1.0,
)

names = lfw.target_names
counts = np.bincount(lfw.target, minlength=len(names))
order = np.argsort(-counts)

print(f'✅ 下載完成')
print(f'   {lfw.images.shape[0]} 張照片，{len(names)} 個人，每張 {lfw.images.shape[1]}×{lfw.images.shape[2]}\n')
print('照片最多的前 15 個人（可以挑其中一個當「要放行的人」）：\n')
for i in order[:15]:
    print(f'   {names[i]:<28} {counts[i]:>4} 張')

正在下載 LFW…（第一次約 1~3 分鐘，之後會用快取）



---
## 3️⃣ 建立資料夾結構

這一格會把照片寫進資料夾。**注意「陌生人」的設計**：

我會**整個人**保留起來，讓他完全不出現在訓練資料裡 ——
不是從「其他人」裡挑一張出來當測試。

**為什麼？** 因為如果陌生人的臉在訓練時出現過（哪怕是被歸類在「其他人」），
那測試就沒有意義了 —— 模型當然認得出他。
**真正的 open-set 測試，測的是「模型沒看過的人」。**

In [ ]:
#@title 🏗️ 建立資料集 { display-mode: "both" }

#@markdown **要放行的人** —— 從上一格的名單挑一個名字複製過來
#@markdown （留空 = 自動挑照片第二多的人；第一名通常是 George W Bush，張數多到誇張）
authorized_name = ""  #@param {type:"string"}

#@markdown **「要放行的人」要幾張照片**
n_authorized = 25  #@param {type:"slider", min:10, max:60, step:5}

#@markdown **「其他人」要幾張照片**（會盡量來自越多不同的人越好）
n_others = 40  #@param {type:"slider", min:20, max:80, step:10}

#@markdown **重建前先清空舊資料**
clear_old = True  #@param {type:"boolean"}

rng = np.random.default_rng(42)

# ── 挑「要放行的人」──
if authorized_name.strip():
    matches = [i for i, n in enumerate(names) if n == authorized_name.strip()]
    assert matches, f'❌ 找不到 {authorized_name}。請從上面那格的名單裡挑一個複製過來'
    ME = matches[0]
else:
    ME = int(order[1])          # 第二多的（第一名通常張數多到誇張）

# ── 挑「陌生人」：整個人保留，不進訓練資料 ──
STRANGER = int(order[2]) if order[2] != ME else int(order[3])

print(f'👤 要放行的人：{names[ME]}（{counts[ME]} 張）')
print(f'🕵️ 陌生人　　：{names[STRANGER]}（整個人保留，完全不進訓練資料）\n')

# ── 清空舊資料 ──
if clear_old:
    for d in (FACES_DIR, TEST_DIR):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)

def save_img(arr_rgb_float, path, size=300):
    """LFW 給的是 float32 RGB [0,1]。要轉成 uint8 BGR 才能給 cv2 存。
    順便放大到 300×300 —— 原圖只有 154×154，太小的話人臉偵測器會抓不到。
    """
    img = (np.clip(arr_rgb_float, 0, 1) * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)
    cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])

# ── ① 要放行的人 ──
me_idx = np.where(lfw.target == ME)[0]
rng.shuffle(me_idx)
me_test, me_train = me_idx[0], me_idx[1:1 + n_authorized]   # 第一張留給測試

me_dir = os.path.join(FACES_DIR, names[ME].replace(' ', '_'))
os.makedirs(me_dir, exist_ok=True)
for k, i in enumerate(me_train):
    save_img(lfw.images[i], os.path.join(me_dir, f'{k:03d}.jpg'))

# ── ② 其他人（排除「要放行的人」和「陌生人」）──
other_idx = np.where((lfw.target != ME) & (lfw.target != STRANGER))[0]
# 盡量讓每個人只出幾張，才會夠多樣 —— 這是「其他人」品質的關鍵
by_person = {}
for i in other_idx:
    by_person.setdefault(int(lfw.target[i]), []).append(i)

picked, per_person = [], 2
for person_ids in rng.permutation(list(by_person.keys())):
    picked += by_person[int(person_ids)][:per_person]
    if len(picked) >= n_others:
        break
picked = picked[:n_others]

others_dir = os.path.join(FACES_DIR, 'others')
os.makedirs(others_dir, exist_ok=True)
for k, i in enumerate(picked):
    save_img(lfw.images[i], os.path.join(others_dir, f'{k:03d}.jpg'))

n_distinct = len({int(lfw.target[i]) for i in picked})

# ── ③ 測試照 ──
save_img(lfw.images[me_test], os.path.join(TEST_DIR, 'demo.jpg'))

stranger_idx = np.where(lfw.target == STRANGER)[0]
save_img(lfw.images[stranger_idx[0]], os.path.join(TEST_DIR, 'stranger.jpg'))

print('✅ 建好了\n')
print(f'   faces/{names[ME].replace(" ", "_"):<24} {len(me_train):>3} 張')
print(f'   faces/others{"":<19} {len(picked):>3} 張（來自 {n_distinct} 個不同的人）')
print(f'   test/demo.jpg{"":<18} 1 張　← {names[ME]}，訓練時沒看過')
print(f'   test/stranger.jpg{"":<14} 1 張　← {names[STRANGER]}，完全沒進過訓練資料')
print(f'\n👉 放行名單請填：{names[ME].replace(" ", "_")}')

---
## 4️⃣ 檢查：人臉偵測抓得到嗎？

**這一格很重要，不要跳過。**

主 notebook 會用 Haar Cascade 把臉切出來。如果這裡偵測率太低，
那邊的資料就會不夠用。先在這裡確認。

In [ ]:
detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def imread_unicode(path):
    return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)

def detect_ok(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    return len(faces) > 0

print('人臉偵測率：\n')
total_ok = total_n = 0
for d in sorted(os.listdir(FACES_DIR)):
    folder = os.path.join(FACES_DIR, d)
    if not os.path.isdir(folder):
        continue
    files = sorted(glob.glob(os.path.join(folder, '*.jpg')))
    ok = sum(detect_ok(imread_unicode(f)) for f in files)
    total_ok += ok; total_n += len(files)
    rate = ok / len(files) * 100 if files else 0
    flag = '✅' if rate >= 70 else '⚠️'
    print(f'  {flag} {d:<24} {ok:>3} / {len(files):<3}  ({rate:.0f}%)')

for f in sorted(glob.glob(os.path.join(TEST_DIR, '*.jpg'))):
    ok = detect_ok(imread_unicode(f))
    print(f'  {"✅" if ok else "❌"} test/{os.path.basename(f):<18} {"偵測到臉" if ok else "偵測不到！換一張"}')

print(f'\n整體偵測率：{total_ok}/{total_n} ({total_ok/total_n*100:.0f}%)')
if total_ok / max(total_n, 1) < 0.7:
    print('⚠️ 偵測率偏低。回上一格把 n_authorized / n_others 調高一點，')
    print('   或換一個人試試（有些人的照片角度比較刁鑽）。')

### 看一下照片長什麼樣

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
me_name = [d for d in os.listdir(FACES_DIR)
           if os.path.isdir(os.path.join(FACES_DIR, d)) and d != 'others'][0]

for row, folder in enumerate([me_name, 'others']):
    files = sorted(glob.glob(os.path.join(FACES_DIR, folder, '*.jpg')))[:6]
    for col in range(6):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(files):
            ax.imshow(cv2.cvtColor(imread_unicode(files[col]), cv2.COLOR_BGR2RGB))
        if col == 0:
            ax.set_title(folder, fontsize=11, loc='left')

# 圖表標題一律用英文 —— Colab 預設沒有中文字型，中文會變成豆腐字 □□□
plt.suptitle('Top: authorized person (same face)     '
             'Bottom: others (all different people)', fontsize=12)
plt.tight_layout(); plt.show()